# 6-hour Solar Wind Speed Prediction — Baseline

이 Baseline 노트북 하나로 데이터 로드, train/validation 분리, GPU 학습,
validation 평가, test 추론과 최종 제출 본인 `submission.csv` 생성을 수행합니다.

- 입력: 과거 5일의 193 Å/211 Å 영상 20쌍과 태양풍 속도 20개(6시간 간격)
- 출력: 마지막 입력 이후 6~72시간 태양풍 속도 12개(6시간 간격)


In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

# Python, NumPy, PyTorch의 seed를 같게 설정
SEED = 777
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# notebook과 같은 위치의 기본 데이터/출력 경로를 사용합니다.
DATA_ROOT = Path("public_dataset/competition_dataset_6h")
OUTPUT_DIR = Path("outputs/baseline_6h")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 64
CHANNELS = ("193", "211")
RMSE_EPSILON = 1e-8
BATCH_SIZE = 256
EPOCHS = 20
NUM_WORKERS = 4
# CUDA를 사용할 수 있으면 GPU와 mixed precision(AMP)을 자동으로 사용합니다.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: CUDA Unavailable")

print("PyTorch:", torch.__version__)
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("data:", DATA_ROOT.resolve())


## 1. Dataload

In [ ]:
IMAGE_COLUMNS = [f"image_{index:02d}" for index in range(20)]
WIND_COLUMNS = [f"wind_{index:02d}" for index in range(20)]
TARGET_COLUMNS = [f"target_{index:02d}" for index in range(12)]

# 세 split은 미리 분리되어 있으며 notebook에서 다시 나누지 않습니다.
info = json.loads((DATA_ROOT / "dataset_info.json").read_text(encoding="utf-8"))
train_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")
train_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")
val_inputs = pd.read_csv(DATA_ROOT / "validation/inputs.csv")
val_targets_frame = pd.read_csv(DATA_ROOT / "validation/targets.csv")
test_inputs = pd.read_csv(DATA_ROOT / "test/inputs.csv")
test_ids = pd.read_csv(DATA_ROOT / "test/test_ids.csv")

# 입력과 정답의 sample 순서, ID 중복, split 간 누락/혼입을 학습 전에 검사합니다.
assert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()
assert val_inputs.sample_id.tolist() == val_targets_frame.sample_id.tolist()
assert test_inputs.sample_id.tolist() == test_ids.sample_id.tolist()
assert train_inputs.sample_id.is_unique
assert val_inputs.sample_id.is_unique
assert test_inputs.sample_id.is_unique
assert set(train_inputs.sample_id).isdisjoint(val_inputs.sample_id)
assert set(train_inputs.sample_id).isdisjoint(test_inputs.sample_id)
assert set(val_inputs.sample_id).isdisjoint(test_inputs.sample_id)
assert set(IMAGE_COLUMNS + WIND_COLUMNS).issubset(train_inputs.columns)
assert set(IMAGE_COLUMNS + WIND_COLUMNS).issubset(val_inputs.columns)
assert set(TARGET_COLUMNS).issubset(train_targets_frame.columns)
assert set(TARGET_COLUMNS).issubset(val_targets_frame.columns)
assert not any(column.startswith("target_") for column in test_inputs.columns)

# 모델 입력에 바로 사용할 수 있도록 target과 row index를 NumPy 배열로 준비합니다.
train_targets = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)
val_targets = val_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)
train_index = np.arange(len(train_inputs), dtype=np.int64)
val_index = np.arange(len(val_inputs), dtype=np.int64)
test_index = np.arange(len(test_inputs), dtype=np.int64)

print(json.dumps(info["counts"], ensure_ascii=False, indent=2))
print("train/validation/test samples:", len(train_index), len(val_index), len(test_index))


## 2. 64×64 image memory-map 전처리와 DataLoader 정의

매 epoch 마다 resize 를 수행하게 되면 불 필요하게 학습시간이 늘어나게 됩니다. 따라서 이미지들은 최초 실행에서 모델의 인풋 형태인 64×64 grayscale `uint8` 배열로 변환합니다.

이후 실행에서는 동일한 이미지 크기, 채널 및 파일 목록의 cache가 존재하면 기존 NumPy memory-map을 재사용합니다.

In [ ]:
def prepare_image_memmap(split, inputs):
    # 같은 inputs.csv로 만든 정상 cache가 있으면 PNG 전처리를 반복하지 않습니다.
    image_root = DATA_ROOT / split
    cache_root = DATA_ROOT / "resized_cache" / f"{IMAGE_SIZE}px"
    cache_root.mkdir(parents=True, exist_ok=True)
    array_path = cache_root / f"{split}_images.npy"
    metadata_path = cache_root / f"{split}_metadata.json"
    # 여러 sample에서 반복되는 PNG는 고유 파일당 한 번만 resize해 저장합니다.
    filenames = sorted(pd.unique(inputs[IMAGE_COLUMNS].to_numpy().ravel()).tolist())
    expected = {
        "image_size": IMAGE_SIZE,
        "channels": list(CHANNELS),
        "filenames": filenames,
    }
    valid = False
    if array_path.exists() and metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            cached = np.load(array_path, mmap_mode="r")
            valid = metadata == expected and cached.shape == (
                len(filenames), len(CHANNELS), IMAGE_SIZE, IMAGE_SIZE
            ) and cached.dtype == np.uint8
        except (OSError, ValueError, json.JSONDecodeError):
            valid = False
    if not valid:
        # partial 파일에 먼저 완성한 뒤 이름을 바꿔 중단된 cache의 재사용을 막습니다.
        array_temp = array_path.with_name(array_path.name + f".partial.{os.getpid()}")
        metadata_temp = metadata_path.with_name(
            metadata_path.name + f".partial.{os.getpid()}"
        )
        # uint8 memory-map은 전체 이미지를 RAM에 올리지 않고 필요한 부분만 읽습니다.
        resized_images = np.lib.format.open_memmap(
            array_temp, mode="w+", dtype=np.uint8,
            shape=(len(filenames), len(CHANNELS), IMAGE_SIZE, IMAGE_SIZE),
        )
        resampling = Image.Resampling.BILINEAR
        for index, filename in enumerate(filenames):
            for channel_index, channel in enumerate(CHANNELS):
                with Image.open(image_root / channel / filename) as image:
                    resized_images[index, channel_index] = np.asarray(
                        image.convert("L").resize(
                            (IMAGE_SIZE, IMAGE_SIZE), resampling
                        ), dtype=np.uint8,
                    )
            if (index + 1) % 2000 == 0 or index + 1 == len(filenames):
                print(f"{split} resize: {index + 1}/{len(filenames)}", flush=True)
        resized_images.flush()
        del resized_images
        metadata_temp.write_text(
            json.dumps(expected, ensure_ascii=False) + "\n", encoding="utf-8"
        )
        array_temp.replace(array_path)
        metadata_temp.replace(metadata_path)
        print(f"created resized cache: {array_path.resolve()}")
    else:
        print(f"reusing resized cache: {array_path.resolve()}")
    image_array = np.load(array_path, mmap_mode="r")
    image_index = {filename: index for index, filename in enumerate(filenames)}
    return image_array, image_index


train_image_array, train_image_index = prepare_image_memmap("train", train_inputs)
val_image_array, val_image_index = prepare_image_memmap("validation", val_inputs)
test_image_array, test_image_index = prepare_image_memmap("test", test_inputs)


class SolarWindDataset(Dataset):
    def __init__(self, image_array, image_index, inputs, indexes, targets=None):
        self.image_array = image_array
        # CSV의 파일명을 cache row 번호로 한 번만 변환해 epoch별 문자열 탐색을 없앱니다.
        self.image_indexes = np.asarray([
            [image_index[filename] for filename in row]
            for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)
        ], dtype=np.int32)
        # wind와 target은 약 0.x 규모로 맞춰 학습을 안정화하고, 평가 때 km/s로 복원합니다.
        self.wind = inputs[WIND_COLUMNS].to_numpy(np.float32) / 1000.0
        self.sample_ids = inputs.sample_id.to_numpy()
        self.indexes = np.asarray(indexes, dtype=np.int64)
        self.targets = targets

    def __len__(self):
        return len(self.indexes)

    def __getitem__(self, item):
        row_index = int(self.indexes[item])
        # 반환 shape: images=(20,2,64,64), wind=(20,), target=(12,)
        images = np.asarray(
            self.image_array[self.image_indexes[row_index]], dtype=np.float32
        ) / 255.0
        result = {
            "images": torch.from_numpy(images),
            "wind": torch.from_numpy(self.wind[row_index]),
            "sample_id": self.sample_ids[row_index],
        }
        if self.targets is not None:
            result["target"] = torch.from_numpy(
                np.asarray(self.targets[row_index], dtype=np.float32) / 1000.0
            )
        return result


def seed_worker(worker_id):
    worker_seed = (SEED + worker_id) % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_loader(dataset, shuffle):
    options = dict(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED),
    )
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)


train_dataset = SolarWindDataset(
    train_image_array, train_image_index, train_inputs, train_index, train_targets
)
val_dataset = SolarWindDataset(
    val_image_array, val_image_index, val_inputs, val_index, val_targets
)
train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset, shuffle=False)

batch = next(iter(train_loader))
assert batch["images"].shape[1:] == (20, 2, 64, 64)
assert batch["wind"].shape[1:] == (20,)
assert batch["target"].shape[1:] == (12,)
print(batch["images"].shape, batch["wind"].shape, batch["target"].shape)


## 3. 3D CNN + Inception + LSTM baseline

인풋 Tensor는 `(batch, time, channel, height, width)`로 정의됩니다.


In [ ]:
class Inception3D(nn.Module):
    def __init__(self, in_channels, branch_channels=32):
        super().__init__()
        self.branch_1 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True)
        )
        self.branch_3 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True),
            nn.Conv3d(
                branch_channels, branch_channels, (1, 3, 3),
                padding=(0, 1, 1)
            ),
            nn.ReLU(inplace=True),
        )
        self.branch_5 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True),
            nn.Conv3d(
                branch_channels, branch_channels, (1, 5, 5),
                padding=(0, 2, 2)
            ),
            nn.ReLU(inplace=True),
        )
        self.branch_pool = nn.Sequential(
            nn.MaxPool3d((1, 3, 3), stride=1, padding=(0, 1, 1)),
            nn.Conv3d(in_channels, branch_channels, 1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return torch.cat(
            [
                self.branch_1(x), self.branch_3(x),
                self.branch_5(x), self.branch_pool(x),
            ],
            dim=1,
        )


class SolarWindBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(2, 32, (1, 5, 5), padding=(0, 2, 2)),
            nn.ReLU(inplace=True),
            nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),
        )
        blocks = []
        in_channels = 32
        for _ in range(3):
            blocks.extend(
                [
                    Inception3D(in_channels, 32),
                    nn.MaxPool3d(
                        (1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)
                    ),
                ]
            )
            in_channels = 128
        self.image_encoder = nn.Sequential(*blocks)
        self.image_lstm = nn.LSTM(
            input_size=128 * 4 * 4,
            hidden_size=128,
            batch_first=True,
        )
        self.wind_encoder = nn.Sequential(
            nn.Linear(20, 128), nn.SELU(inplace=True),
            nn.Linear(128, 64), nn.SELU(inplace=True),
        )
        self.head = nn.Sequential(
            nn.Linear(128 + 64, 64), nn.ReLU(inplace=True), nn.Linear(64, 12)
        )

    def forward(self, images, wind):
        # DataLoader의 (B,T,C,H,W)를 Conv3d 입력인 (B,C,T,H,W)로 바꿉니다.
        image_features = images.permute(0, 2, 1, 3, 4).contiguous()
        image_features = self.stem(image_features)
        image_features = self.image_encoder(image_features)
        # 시점별 공간 특징을 펼쳐 (B,T,C*H*W) 형태로 LSTM에 전달합니다.
        image_features = image_features.permute(0, 2, 1, 3, 4).flatten(2)
        _, (hidden, _) = self.image_lstm(image_features)
        image_features = F.relu(hidden[-1])
        wind_features = self.wind_encoder(wind)
        return self.head(torch.cat([image_features, wind_features], dim=1))


model = SolarWindBaseline().to(DEVICE)
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print("trainable parameters:", f"{trainable_parameters:,}")


## 4. GPU 학습


In [ ]:
# 매 학습 실행마다 새 모델을 만들며 pretrained/이전 실행 weight는 불러오지 않습니다.
# ※ 참가자 분들은 이전 학습 결과가 덮어 저장되는 경우를 방지하기 위해서 각자의 버전관리 시스템을 적용하여 이 섹션을 수정해서 사용하시길 바랍니다.
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
model = SolarWindBaseline().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.25, patience=1, min_lr=1e-6
)
# AMP는 GPU 메모리 사용량을 줄이고 연산을 가속합니다.
scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
checkpoint_path = OUTPUT_DIR / "best_model.pth"
if checkpoint_path.exists():
    checkpoint_path.unlink()
print("initialized model from scratch; removed any previous checkpoint")
patience = 5
best_val_rmse = float("inf")
epochs_without_improvement = 0
history = []


def run_epoch(loader, training):
    model.train(training)
    squared_error_sum = 0.0
    value_count = 0
    for batch in loader:
        images = batch["images"].to(
            DEVICE, non_blocking=PIN_MEMORY
        )
        wind = batch["wind"].to(DEVICE, non_blocking=PIN_MEMORY)
        target = batch["target"].to(DEVICE, non_blocking=PIN_MEMORY)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                prediction = model(images, wind)
                # /1000 정규화된 단위에서 batch RMSE를 학습 loss로 사용합니다.
                mse = F.mse_loss(prediction, target)
                loss = torch.sqrt(mse + RMSE_EPSILON)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        # 사용자에게 익숙한 km/s 단위로 되돌려 전체 원소 기준 RMSE를 누적합니다.
        error_km_s = (prediction.detach() - target) * 1000.0
        squared_error_sum += float(torch.sum(error_km_s ** 2).cpu())
        value_count += error_km_s.numel()
    return math.sqrt(squared_error_sum / value_count)


for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    train_rmse = run_epoch(train_loader, training=True)
    with torch.no_grad():
        val_rmse = run_epoch(val_loader, training=False)
    scheduler.step(val_rmse)
    learning_rate = optimizer.param_groups[0]["lr"]
    elapsed = time.perf_counter() - started
    row = {
        "epoch": epoch,
        "train_rmse_km_s": train_rmse,
        "val_rmse_km_s": val_rmse,
        "learning_rate": learning_rate,
        "seconds": elapsed,
    }
    history.append(row)
    print(
        f"epoch={epoch:03d} train_rmse={train_rmse:.3f} "
        f"val_rmse={val_rmse:.3f} lr={learning_rate:.2e} "
        f"seconds={elapsed:.1f}"
    )
    # validation RMSE가 가장 낮은 epoch만 checkpoint로 보관합니다.
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "val_rmse_km_s": val_rmse,
                "channels": CHANNELS,
                "initialization": "random_from_scratch",
            },
            checkpoint_path,
        )
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print("early stopping")
            break

history_frame = pd.DataFrame(history)
history_frame.to_csv(OUTPUT_DIR / "history.csv", index=False)
history_frame.plot(
    x="epoch", y=["train_rmse_km_s", "val_rmse_km_s"], grid=True
)
plt.ylabel("RMSE (km/s)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "learning_curve.png", dpi=140)
plt.show()

checkpoint = torch.load(
    checkpoint_path, map_location=DEVICE, weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"])
print("loaded best epoch:", checkpoint["epoch"])


## 5. Validation 데이터 평가


In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval()
    predictions = []
    sample_ids = []
    for batch in loader:
        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind = batch["wind"].to(DEVICE, non_blocking=PIN_MEMORY)
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            output = model(images, wind)
        predictions.append(output.float().cpu().numpy() * 1000.0)
        sample_ids.extend(batch["sample_id"])
    return np.concatenate(predictions), sample_ids


validation_prediction, validation_ids = predict(val_loader)
validation_target = np.asarray(val_targets[val_index], dtype=np.float64)
assert validation_ids == val_inputs.iloc[val_index].sample_id.tolist()

# persistence baseline: 마지막 관측 wind가 72시간 동안 유지된다고 가정하는 baseline 객관적 성능 비교를 위해 사용 가능 (유의미한 모델이 학습되었는지 판단하기 위함)
validation_persistence = np.repeat(
    val_inputs.iloc[val_index][[WIND_COLUMNS[-1]]].to_numpy(np.float64),
    12,
    axis=1,
)


def metrics_by_horizon(y_true, y_pred):
    rows = []
    for index in range(12):
        actual = y_true[:, index]
        predicted = y_pred[:, index]
        error = predicted - actual
        denominator = np.std(actual) * np.std(predicted)
        correlation = (
            float(np.corrcoef(actual, predicted)[0, 1])
            if denominator > 0 else np.nan
        )
        rows.append({
            "horizon_hours": (index + 1) * 6,
            "rmse_km_s": float(np.sqrt(np.mean(error ** 2))),
            "mae_km_s": float(np.mean(np.abs(error))),
            "correlation": correlation,
        })
    return pd.DataFrame(rows)


validation_metrics = metrics_by_horizon(
    validation_target, validation_prediction
)
persistence_metrics = metrics_by_horizon(
    validation_target, validation_persistence
)
validation_metrics["persistence_rmse_km_s"] = persistence_metrics.rmse_km_s
validation_metrics.to_csv(
    OUTPUT_DIR / "validation_metrics.csv", index=False
)
print(
    "overall validation RMSE:",
    float(np.sqrt(np.mean((validation_prediction - validation_target) ** 2))),
)
validation_metrics

## 6. Test 추론 및 submission.csv 생성


In [ ]:
# test 추론 전에 학습용 객체와 CUDA cache를 정리해 불필요한 메모리를 반환합니다.
del train_loader, val_loader, train_dataset, val_dataset
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

test_dataset = SolarWindDataset(
    test_image_array, test_image_index, test_inputs, test_index, targets=None
)
test_loader = make_loader(test_dataset, shuffle=False)
test_prediction, predicted_ids = predict(test_loader)
expected_ids = test_inputs.iloc[test_index].sample_id.tolist()
assert predicted_ids == expected_ids
assert test_prediction.shape == (len(test_index), 12)
assert np.isfinite(test_prediction).all()

# 제출 형식은 sample_id 다음에 target_00~target_11이 오는 13개 column입니다.
submission = pd.DataFrame(test_prediction, columns=TARGET_COLUMNS)
submission.insert(0, "sample_id", predicted_ids)
submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

assert submission.columns.tolist() == ["sample_id"] + TARGET_COLUMNS
assert submission.sample_id.is_unique

print("saved:", submission_path.resolve())
print("shape:", submission.shape)

del test_dataset, test_loader
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

submission.head()